# Silver - Atendimento Ocorrências

Limpeza e padronização dos dados de atendimento e ocorrências.

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
sistema = 'case'
table_name = 'atendimento_ocorrencias'
input_path = f"{var_bronze}/{sistema}/{table_name}/data"
output_path_data = f"{var_silver}/{sistema}/{table_name}/data"
table_name_schema = f'{var_environment}.{var_silver_schema}.{sistema}_{table_name}'

In [ ]:
from pyspark.sql.functions import col, to_timestamp, row_number
from pyspark.sql.window import Window

df_bronze = spark.read.format("delta").load(input_path)

# Limpeza e Deduplicação (mantendo o evento de maior data/hora por ticket)
window_spec = Window.partitionBy("ticket_id").orderBy(col("created_at").desc())

df_clean = (
    df_bronze
    .withColumn("created_at", to_timestamp(col("created_at")))
    .withColumn("rn", row_number().over(window_spec))
    .filter("rn = 1")
    .drop("rn")
    .select(
        col("ticket_id").cast("string").alias("id_ticket"),
        col("order_id").cast("string").alias("id_pedido"),
        col("event_type").cast("string").alias("tipo_evento"),
        col("created_at").alias("data_criacao"),
        col("severity").cast("string").alias("severidade"),
        col("status").cast("string").alias("status_ticket")
    )
    .filter(col("id_ticket").isNotNull())
)

In [ ]:
process_data(
    df_write=df_clean,
    tipo_carga='delta',
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    chave_clusterby=['tipo_evento'],
    chave_upsert='id_ticket'
)